# Notebook 5 — Prediction Comparison (the controlled comparison)
### Leakage-clean pollutant regression: GNN vs RNN / LSTM / GRU vs Xuanzhe

This is the **comparison** that complements the detection headline. It answers the Xuanzhe-style
question — *can we predict the 9 pollutant concentrations?* — but **without the leakage** in the
earlier pipeline, and with the **ontology actually driving** the graph model.

- **Targets**: the 9 pollutants (`co2, so2, no2, co, o3, tvoc, pm2_5, pm10, pm1`), in real units.
- **Inputs (leakage-clean)**: only the **context** variables (climate + acoustic + humidity-rate).
  The particle counts, `TypPS`, `dCO2dt`, `health`, `performance` are excluded — they trivially encode
  the targets (PM mass *is* computed from the count bins).
- **Two model families on identical splits/scalers**:
  - **GNN** (`GCN`/`GIN`/`SAT-Graph`) — *relational*: the pollutant nodes are masked at input and
    predicted from the context nodes through the ontology edges.
  - **Sequential** (`RNN`/`LSTM`/`GRU`) — *temporal*: a sliding window of past context features.
- **Metrics**: per-pollutant and average **R² / MAE / RMSE** (predictions inverse-transformed to real
  units), then placed against Xuanzhe.

Honest framing for the write-up: this is **prediction** (concentration regression), distinct from the
detection taux; and the GNN exploits relational structure within a window while the RNN exploits
temporal history — both from the same leakage-clean inputs.


In [ ]:
import json, time, copy, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader as TDataLoader
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, GINConv, TransformerConv, BatchNorm, LayerNorm

warnings.filterwarnings("ignore")
SEED = 42; torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE)

## 1. Load the graph substrate + prediction design (Notebook 2)
Notebook 2 already prepared the leakage-clean prediction split (normal windows only, per-deployment
temporal), the target/context node indices, the targets in real units (`Y_raw`) and standardised
(`Y_std`), and the target scaler (`Y_mu`/`Y_sd`).


In [ ]:
_C = [Path("processed/graph"), Path("processed"), Path(".")]
GP = next((p for p in _C if (p / "graph_windows.npz").exists()), Path("processed/graph"))
d = np.load(GP / "graph_windows.npz", allow_pickle=True)
man = json.load(open(GP / "graph_manifest.json"))

X = d["X"].astype(np.float32)                  # (W, N, F) pooled-standardised inputs
edge_index_np = d["edge_index"]
target_idx = d["target_idx"]; context_idx = d["context_idx"]
Y_raw = d["Y_raw"].astype(np.float32); Y_std = d["Y_std"].astype(np.float32)
Y_mu = d["Y_mu"]; Y_sd = d["Y_sd"]
pred_train = d["pred_train"]; pred_val = d["pred_val"]; pred_test = d["pred_test"]
location = d["location"].astype(str)
W, N_NODES, F_CH = X.shape
TARGETS = man["prediction"]["targets"]; CONTEXT = man["prediction"]["context_inputs"]
print(f"Windows={W} Nodes={N_NODES} | targets={len(TARGETS)} context={len(CONTEXT)}")
print(f"pred split train/val/test = {pred_train.sum()}/{pred_val.sum()}/{pred_test.sum()} (normal only)")

## 2. Mask the inputs (leakage control) + shared metrics
Only the **context** nodes keep their features; target and excluded nodes are zeroed (= their
standardised mean). The graph still routes information context→pollutant through the ontology edges.


In [ ]:
keep = np.zeros(N_NODES, bool); keep[context_idx] = True
X_pred = X.copy()
X_pred[:, ~keep, :] = 0.0                        # mask targets + excluded nodes
print(f"Masked input: {keep.sum()} context nodes visible, {(~keep).sum()} nodes zeroed.")

def inv(std):                                    # standardised -> real units
    return std * Y_sd + Y_mu
def r2(yt, yp):
    ss = ((yt - yp) ** 2).sum(); tot = ((yt - yt.mean()) ** 2).sum(); return float(1 - ss / max(tot, 1e-12))
def mae(yt, yp): return float(np.abs(yt - yp).mean())
def rmse(yt, yp): return float(np.sqrt(((yt - yp) ** 2).mean()))

def per_pollutant_metrics(name, true_real, pred_real, family):
    rr = {"Model": name}; rm = {"Model": name}; rt = {"Model": name}
    for i, pol in enumerate(TARGETS):
        rr[pol] = round(r2(true_real[:, i], pred_real[:, i]), 4)
        rm[pol] = round(mae(true_real[:, i], pred_real[:, i]), 4)
        rt[pol] = round(rmse(true_real[:, i], pred_real[:, i]), 4)
    for dct in (rr, rm, rt): dct["AVG"] = round(float(np.mean([dct[p] for p in TARGETS])), 4)
    return rr, rm, rt, {"Model": name, "Family": family,
                        "R2_avg": rr["AVG"], "MAE_avg": rm["AVG"], "RMSE_avg": rt["AVG"]}
print("Masking + metric helpers ready.")

## 3. GNN predictors — masked pollutant-node regression
Each window is a 30-node graph (context features only + a learned node-id embedding so the model knows
which node is which). The encoder is GCN / GIN / SAT-Graph; a shared head reads the **pollutant
nodes'** embeddings and outputs their 9 concentrations.


In [ ]:
edge_index = torch.tensor(edge_index_np, dtype=torch.long)
def degree_pe(ei, n):
    deg = torch.zeros(n); deg.scatter_add_(0, ei[0], torch.ones(ei.size(1)))
    return (deg / max(deg.max().item(), 1.0)).unsqueeze(1)
def rwse_pe(ei, n, k=8):
    deg = torch.zeros(n); deg.scatter_add_(0, ei[0], torch.ones(ei.size(1)))
    val = (1.0 / (deg + 1e-8))[ei[0]]; P = torch.sparse_coo_tensor(ei, val, (n, n)).coalesce().to_dense()
    rw = torch.zeros(n, k); Pk = P.clone()
    for i in range(k): rw[:, i] = Pk.diagonal(); Pk = Pk @ P
    return rw
PE = torch.cat([degree_pe(edge_index, N_NODES), rwse_pe(edge_index, N_NODES, 8)], dim=1)

def make_data(mask):
    idx = np.where(mask)[0]
    ds = [Data(x=torch.tensor(X_pred[w], dtype=torch.float), edge_index=edge_index,
               y=torch.tensor(Y_std[w], dtype=torch.float).unsqueeze(0)) for w in idx]
    return ds, idx
g_train, _ = make_data(pred_train); g_val, _ = make_data(pred_val); g_test, te_idx = make_data(pred_test)
gtr = DataLoader(g_train, batch_size=128, shuffle=True)
gva = DataLoader(g_val, batch_size=256, shuffle=False)
gte = DataLoader(g_test, batch_size=256, shuffle=False)

class PredGNN(nn.Module):
    def __init__(self, in_ch, n_nodes, target_idx, hid=64, conv="gcn", nl=2, heads=4, dr=0.2, id_dim=8, pe=None):
        super().__init__()
        self.conv_type = conv; self.dr = dr; self.n_nodes = n_nodes
        self.register_buffer("tidx", torch.tensor(np.asarray(target_idx), dtype=torch.long))
        self.node_emb = nn.Embedding(n_nodes, id_dim)
        extra = id_dim + (pe.shape[1] if (conv == "sat" and pe is not None) else 0)
        if conv == "sat": self.register_buffer("pe", pe)
        self.enc_in = nn.Linear(in_ch + extra, hid)
        self.convs = nn.ModuleList(); self.norms = nn.ModuleList()
        for _ in range(nl):
            if conv == "gcn":
                self.convs.append(GCNConv(hid, hid, add_self_loops=True)); self.norms.append(BatchNorm(hid))
            elif conv == "gin":
                mlp = nn.Sequential(nn.Linear(hid, 2 * hid), nn.BatchNorm1d(2 * hid), nn.ReLU(), nn.Linear(2 * hid, hid))
                self.convs.append(GINConv(mlp, train_eps=True)); self.norms.append(BatchNorm(hid))
            else:
                self.convs.append(TransformerConv(hid, hid // heads, heads=heads, dropout=dr, concat=True))
                self.norms.append(LayerNorm(hid))
        self.head = nn.Sequential(nn.Linear(hid, hid // 2), nn.ReLU(), nn.Dropout(dr), nn.Linear(hid // 2, 1))

    def forward(self, x, ei):
        B = x.size(0) // self.n_nodes
        ids = torch.arange(self.n_nodes, device=x.device).repeat(B)
        parts = [x, self.node_emb(ids)]
        if self.conv_type == "sat": parts.append(self.pe.repeat(B, 1))
        h = torch.cat(parts, dim=-1)
        h = F.relu(self.enc_in(h)); h = F.dropout(h, self.dr, self.training)
        for c, n in zip(self.convs, self.norms):
            h = F.relu(n(c(h, ei))); h = F.dropout(h, self.dr, self.training)
        h = h.view(B, self.n_nodes, -1)[:, self.tidx, :]      # (B, 9, hid)
        return self.head(h).squeeze(-1)                        # (B, 9)
print("PredGNN defined.")

## 4. Train the GNN predictors

In [ ]:
def train_graph(model, name, epochs=150, patience=20, lr=1e-3):
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=10, min_lr=1e-6)
    crit = nn.MSELoss(); best = 1e9; bs = None; pc = 0; t0 = time.time()
    print(f"\n=== {name} ===")
    for ep in range(1, epochs + 1):
        model.train()
        for b in gtr:
            b = b.to(DEVICE); opt.zero_grad()
            loss = crit(model(b.x, b.edge_index), b.y); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        model.eval(); vl = []
        with torch.no_grad():
            for b in gva: b = b.to(DEVICE); vl.append(crit(model(b.x, b.edge_index), b.y).item())
        v = float(np.mean(vl)); sch.step(v)
        if v < best - 1e-6: best = v; bs = copy.deepcopy(model.state_dict()); pc = 0
        else: pc += 1
        if ep % 30 == 0 or ep == 1: print(f"  ep{ep:3d} val_mse={v:.4f}")
        if pc >= patience: print(f"  early stop @ {ep}"); break
    model.load_state_dict(bs); print(f"  done {time.time()-t0:.1f}s")
    return model

@torch.no_grad()
def predict_graph(model):
    model.eval(); P = []
    for b in gte: b = b.to(DEVICE); P.append(model(b.x, b.edge_index).cpu().numpy())
    return inv(np.vstack(P))                       # real units, aligned to te_idx

GCONF = {"GCN": dict(conv="gcn"), "GIN": dict(conv="gin"), "SAT-Graph": dict(conv="sat", heads=4)}
rows_r2, rows_mae, rows_rmse, master = [], [], [], []
Y_true_g = Y_raw[te_idx]
for name, kw in GCONF.items():
    pe = PE if kw["conv"] == "sat" else None
    m = PredGNN(F_CH, N_NODES, target_idx, hid=64, nl=2, pe=pe, **kw)
    m = train_graph(m, name); torch.save(m.state_dict(), GP / f"{name.lower().replace('-','_')}_pred.pt")
    pr = predict_graph(m)
    rr, rm, rt, ms = per_pollutant_metrics(name, Y_true_g, pr, "GNN")
    rows_r2.append(rr); rows_mae.append(rm); rows_rmse.append(rt); master.append(ms)
    print(f"  {name}: R2_avg={ms['R2_avg']}  RMSE_avg={ms['RMSE_avg']}")

## 5. Sequential baselines — RNN / LSTM / GRU
A sliding window of `SEQ` past windows of **context** features predicts the current 9 pollutants.
Sequences never cross the lab↔apartment boundary, and use the same split, scaler, and targets.


In [ ]:
SEQ = 12                                          # past windows of context (≈ 6 h at 30-min stride)
ctx_feat = X[:, context_idx, :].reshape(W, -1)    # (W, n_context*F)  context inputs only

def build_seq(split_mask):
    Xs, ys, idxs = [], [], []
    for loc in ["laboratory", "apartment"]:
        gi = np.where(location == loc)[0]             # time-ordered global indices
        for k in range(len(gi)):
            p = gi[k]
            if not split_mask[p] or k < SEQ: continue
            Xs.append(ctx_feat[gi[k - SEQ:k]]); ys.append(Y_std[p]); idxs.append(p)
    return (np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32), np.array(idxs))

Xtr, ytr, _ = build_seq(pred_train); Xva, yva, _ = build_seq(pred_val); Xte, yte, te_seq_idx = build_seq(pred_test)
IN = Xtr.shape[2]
tr_dl = TDataLoader(TensorDataset(torch.tensor(Xtr), torch.tensor(ytr)), batch_size=64, shuffle=True)
va_dl = TDataLoader(TensorDataset(torch.tensor(Xva), torch.tensor(yva)), batch_size=128)
te_dl = TDataLoader(TensorDataset(torch.tensor(Xte), torch.tensor(yte)), batch_size=128)
print(f"Sequences — train/val/test: {len(Xtr)}/{len(Xva)}/{len(Xte)} | input dim/step={IN}, seq_len={SEQ}")

class SeqReg(nn.Module):
    def __init__(self, insz, hid, out, cell="LSTM", nl=2, dr=0.3):
        super().__init__(); self.cell = cell
        kw = dict(input_size=insz, hidden_size=hid, num_layers=nl, batch_first=True, dropout=dr if nl > 1 else 0.0)
        self.rnn = nn.RNN(**kw, nonlinearity="tanh") if cell == "RNN" else nn.LSTM(**kw) if cell == "LSTM" else nn.GRU(**kw)
        self.head = nn.Sequential(nn.Dropout(dr), nn.Linear(hid, hid // 2), nn.ReLU(), nn.Linear(hid // 2, out))
    def forward(self, x):
        h = self.rnn(x)[1][0][-1] if self.cell == "LSTM" else self.rnn(x)[1][-1]
        return self.head(h)

def train_seq(model, name, epochs=120, patience=20, lr=1e-3):
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=8, min_lr=1e-6)
    crit = nn.MSELoss(); best = 1e9; bs = None; pc = 0; t0 = time.time()
    print(f"\n=== {name} ===")
    for ep in range(1, epochs + 1):
        model.train()
        for xb, yb in tr_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE); opt.zero_grad()
            loss = crit(model(xb), yb); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        model.eval(); vl = []
        with torch.no_grad():
            for xb, yb in va_dl: vl.append(crit(model(xb.to(DEVICE)), yb.to(DEVICE)).item())
        v = float(np.mean(vl)); sch.step(v)
        if v < best - 1e-6: best = v; bs = copy.deepcopy(model.state_dict()); pc = 0
        else: pc += 1
        if ep % 30 == 0 or ep == 1: print(f"  ep{ep:3d} val_mse={v:.4f}")
        if pc >= patience: print(f"  early stop @ {ep}"); break
    model.load_state_dict(bs); print(f"  done {time.time()-t0:.1f}s"); return model

@torch.no_grad()
def predict_seq(model):
    model.eval(); P = []
    for xb, _ in te_dl: P.append(model(xb.to(DEVICE)).cpu().numpy())
    return inv(np.vstack(P))

Y_true_s = Y_raw[te_seq_idx]
for cell in ["RNN", "LSTM", "GRU"]:
    m = train_seq(SeqReg(IN, 128, len(TARGETS), cell=cell), cell)
    torch.save(m.state_dict(), GP / f"{cell.lower()}_pred.pt")
    pr = predict_seq(m)
    rr, rm, rt, ms = per_pollutant_metrics(cell, Y_true_s, pr, "Sequential")
    rows_r2.append(rr); rows_mae.append(rm); rows_rmse.append(rt); master.append(ms)
    print(f"  {cell}: R2_avg={ms['R2_avg']}  RMSE_avg={ms['RMSE_avg']}")

## 6. Results — per-pollutant + overall, and the comparison

In [ ]:
df_r2 = pd.DataFrame(rows_r2); df_mae = pd.DataFrame(rows_mae); df_rmse = pd.DataFrame(rows_rmse)
df_master = pd.DataFrame(master)
for df_, n in [(df_r2, "r2"), (df_mae, "mae"), (df_rmse, "rmse")]:
    df_.to_csv(GP / f"nb05_pred_{n}.csv", index=False)
df_master.to_csv(GP / "nb05_pred_metrics.csv", index=False)

print("=== Overall (averaged over 9 pollutants) ===")
print(df_master.to_string(index=False))
print("\n=== R² per pollutant ===")
print(df_r2.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
ALL = df_master["Model"].tolist()
pal = {"GCN": "#378ADD", "GIN": "#1D9E75", "SAT-Graph": "#D85A30", "RNN": "#7E57C2", "LSTM": "#D4537E", "GRU": "#BA7517"}
for ax, (col, lbl, hib) in zip(axes, [("R2_avg", "R² (higher better)", True),
                                      ("MAE_avg", "MAE (lower better)", False),
                                      ("RMSE_avg", "RMSE (lower better)", False)]):
    vals = [df_master[df_master["Model"] == m][col].values[0] for m in ALL]
    bars = ax.bar(range(len(ALL)), vals, color=[pal[m] for m in ALL])
    best = int(np.argmax(vals)) if hib else int(np.argmin(vals))
    bars[best].set_edgecolor("black"); bars[best].set_linewidth(2)
    ax.axvline(2.5, color="grey", ls="--", lw=1)
    ax.set_xticks(range(len(ALL))); ax.set_xticklabels(ALL, rotation=35, ha="right")
    ax.set_title(lbl)
fig.suptitle("Prediction — GNN (left of dashed) vs Sequential (right); black edge = best", y=1.03)
fig.tight_layout(); plt.savefig(GP / "nb05_pred_bars.png", dpi=130, bbox_inches="tight"); plt.show()
print("Saved: nb05_pred_metrics.csv, per-pollutant CSVs, nb05_pred_bars.png")

## 7. Comparison with Xuanzhe (2024)

Xuanzhe's RNN/LSTM/GRU forecast pollutant concentrations and reported RMSE, finding the three
recurrent models broadly comparable for short-horizon prediction with no decisive winner. Our project
**extends** that with an ontology-driven graph model and a **controlled, leakage-clean** comparison on
identical data, features, split, and metrics.

Two honest caveats for the defense: Xuanzhe's exact per-pollutant values are limited, so the
comparison is directional; and our PM numbers are **leakage-clean** (counts excluded), so they will
look weaker than a pipeline that predicts PM from its own particle counts — that lower number is the
*correct* one.


In [ ]:
gnn = df_master[df_master["Family"] == "GNN"]; seq = df_master[df_master["Family"] == "Sequential"]
best_gnn = gnn.loc[gnn["R2_avg"].idxmax()]; best_seq = seq.loc[seq["R2_avg"].idxmax()]
print("=" * 60)
print("  PREDICTION COMPARISON SUMMARY")
print("=" * 60)
print(f"  Best GNN        : {best_gnn['Model']:10s} R2={best_gnn['R2_avg']:.4f}  RMSE={best_gnn['RMSE_avg']:.3f}")
print(f"  Best Sequential : {best_seq['Model']:10s} R2={best_seq['R2_avg']:.4f}  RMSE={best_seq['RMSE_avg']:.3f}")
print(f"  GNN family mean R2 = {gnn['R2_avg'].mean():.4f}   Sequential mean R2 = {seq['R2_avg'].mean():.4f}")
print()
print("  vs Xuanzhe (2024): RNN/LSTM/GRU comparable for short-horizon forecasting; this project adds")
print("  ontology-driven GNNs + a leakage-clean controlled comparison (identical data/features/split).")
json.dump({"gnn_mean_R2": float(gnn["R2_avg"].mean()), "seq_mean_R2": float(seq["R2_avg"].mean()),
           "best_gnn": best_gnn["Model"], "best_seq": best_seq["Model"]},
          open(GP / "nb05_pred_summary.json", "w"), indent=2, default=str)
print("\nSaved: nb05_pred_summary.json")

## 7b. Ready-to-paste comparison table

A single, report-ready view: the averaged R² / MAE / RMSE for all six models, sorted best-first,
printed to the console **and** saved as an image (`nb05_comparison_table.png`) you can drop straight
into the report. GNN rows and sequential rows are shaded differently; the best R² row is bold.


In [ ]:
# ready-to-paste comparison table: console + PNG
tbl = df_master.sort_values("R2_avg", ascending=False).reset_index(drop=True)
disp = tbl.copy()
disp["R2_avg"] = disp["R2_avg"].map(lambda v: f"{v:.3f}")
disp["MAE_avg"] = disp["MAE_avg"].map(lambda v: f"{v:.2f}")
disp["RMSE_avg"] = disp["RMSE_avg"].map(lambda v: f"{v:.2f}")
disp.columns = ["Model", "Family", "R2 (avg)", "MAE (avg)", "RMSE (avg)"]

print("=== Prediction comparison (averaged over the 9 target pollutants) ===")
print(disp.to_string(index=False))

fig, ax = plt.subplots(figsize=(7.4, 0.6 + 0.42 * (len(disp) + 1)))
ax.axis("off")
the = ax.table(cellText=disp.values, colLabels=list(disp.columns), cellLoc="center", loc="center")
the.auto_set_font_size(False); the.set_fontsize(11); the.scale(1, 1.5)
for (r, c), cellobj in the.get_celld().items():
    cellobj.set_edgecolor("#d9d9d9")
    if r == 0:
        cellobj.set_facecolor("#34495E"); cellobj.get_text().set_color("white"); cellobj.get_text().set_weight("bold")
    else:
        fam = tbl.iloc[r - 1]["Family"]
        cellobj.set_facecolor("#EAF2FB" if fam == "GNN" else "#F5EEF1")
        if r - 1 == 0:
            cellobj.get_text().set_weight("bold")
ax.set_title("Pollutant prediction \u2014 GNN vs sequential (avg over 9 pollutants)", fontsize=12, pad=12)
fig.tight_layout()
fig.savefig(GP / "nb05_comparison_table.png", dpi=150, bbox_inches="tight"); plt.show()
print("\nSaved: nb05_comparison_table.png  (paste straight into the report)")

## 8. Hand-off to Notebook 6

The comparison half is done: GCN / GIN / SAT-Graph vs RNN / LSTM / GRU on leakage-clean pollutant
regression, per-pollutant and average R²/MAE/RMSE, against Xuanzhe.

**Next — Notebook 6: final write-up.** Assemble the two stories side by side: the **detection taux**
(NB4, the headline — did the ontology autoencoder catch the fire / dust / outage) and this
**prediction table** (the controlled comparison), with the clear statement that the two use different
metrics for different questions.
